In [1]:
# E0: Native-language baseline
# XLM-RoBERTa fine-tuned directly on the low-resource dataset

import json
import random
import numpy as np
import pandas as pd
import torch

from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    get_linear_schedule_with_warmup
)
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)


In [2]:
import sys
import torch

print("Python path:", sys.executable)
print("Torch path:", torch.__file__)
print("Torch version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

Python path: /home/jtan/myssp/.venv/bin/python
Torch path: /home/jtan/myssp/.venv/lib/python3.13/site-packages/torch/__init__.py
Torch version: 2.11.0+cu128
CUDA Available: True


In [3]:

"""
Configuration
"""

from pathlib import Path

def _project_root():
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "db").is_dir() and (candidate / "notebooks").is_dir():
            return candidate
    raise FileNotFoundError(f"Could not find project root from {start}")

ROOT = _project_root()
MODELS_DIR = ROOT / "models"
MODELS_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "xlm-roberta-base"

TRAIN_DATA_PATH = str(ROOT / "db" / "goemotions" / "train.tsv")
VALID_DATA_PATH = str(ROOT / "db" / "goemotions" / "dev.tsv")
TEST_DATA_PATH = str(ROOT / "db" / "goemotions" / "test.tsv")
EMOTIONS_PATH = str(ROOT / "db" / "goemotions" / "emotions.txt")
EKMAN_PATH = str(ROOT / "db" / "goemotions" / "ekman_mapping.json")

TEXT_COLUMN = "text"
LABEL_COLUMN = "emotion"

MAX_LENGTH = 140
BATCH_SIZE = 216
EPOCHS = 4
LEARNING_RATE = 2e-5
FOCAL_GAMMA = 1.0

RANDOM_SEED = 42

NUM_GPUS = min(2, torch.cuda.device_count())
DEVICE = torch.device("cuda:0" if NUM_GPUS else "cpu")

print("Using device:", DEVICE)
print("GPUs used:", NUM_GPUS)


Using device: cuda:0
GPUs used: 1


In [4]:

"""
Reproducibility
"""

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


set_seed(RANDOM_SEED)



In [5]:

"""
Load datasets
"""

train_df = pd.read_csv(TRAIN_DATA_PATH, sep='\t', header=None, names=["text", "emotion", "userid"])

# Remove rows with missing text/labels
train_df = train_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])

with open(EMOTIONS_PATH, encoding="utf-8") as emotions_file:
    fear_label_id = str([line.strip() for line in emotions_file if line.strip()].index("fear"))

def remove_fear_samples(dataframe):
    emotion_ids = dataframe[LABEL_COLUMN].astype(str).str.split(",")
    return dataframe[~emotion_ids.apply(lambda ids: fear_label_id in ids)].copy()

train_df = remove_fear_samples(train_df)

print("Number of examples:", len(train_df))
print("\nClass distribution:")
print(train_df[LABEL_COLUMN].value_counts())



Number of examples: 43410

Class distribution:
emotion
27            12823
0              2710
4              1873
15             1857
1              1652
              ...  
0,12,13,26        1
1,2,5,17          1
13,14             1
3,9,12            1
0,1,18            1
Name: count, Length: 711, dtype: int64


In [6]:
test_df = pd.read_csv(TEST_DATA_PATH, sep='\t', header=None, names=["text", "emotion", "userid"])

# Remove rows with missing text/labels
test_df = test_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
test_df = remove_fear_samples(test_df)

print("Number of examples:", len(test_df))
print("\nClass distribution:")
print(test_df[LABEL_COLUMN].value_counts())


Number of examples: 5427

Class distribution:
emotion
27         1606
0           348
15          260
4           236
10          195
           ... 
6,17,22       1
2,3,10        1
0,1,22        1
1,20,27       1
10,12         1
Name: count, Length: 275, dtype: int64


In [7]:
valid_df = pd.read_csv(VALID_DATA_PATH, sep='\t', header=None, names=["text", "emotion", "userid"])

# Remove rows with missing text/labels
valid_df = valid_df.dropna(subset=[TEXT_COLUMN, LABEL_COLUMN])
valid_df = remove_fear_samples(valid_df)

print("Number of examples:", len(valid_df))
print("\nClass distribution:")
print(valid_df[LABEL_COLUMN].value_counts())


Number of examples: 5426

Class distribution:
emotion
27            1592
0              326
15             261
4              258
10             212
              ... 
5,9,24           1
4,12             1
0,4,9            1
0,13,15,26       1
0,17,22          1
Name: count, Length: 296, dtype: int64


In [8]:
"""
Encode labels
"""

with open(EMOTIONS_PATH, encoding="utf-8") as emotions_file:
    labels = [line.strip() for line in emotions_file if line.strip()]

with open(EKMAN_PATH, encoding="utf-8") as ekman_file:
    ekman_mapping = json.load(ekman_file)

label2id = {label: index for index, label in enumerate(labels)}
id2label = {index: label for label, index in label2id.items()}

category_labels = list(ekman_mapping)
category_labels.append("neutral")
category2id = {category: index for index, category in enumerate(category_labels)}
id2category = {index: category for category, index in category2id.items()}

emotion_to_category_id = {}
for category, emotions in ekman_mapping.items():
    for emotion in emotions:
        emotion_to_category_id[label2id[emotion]] = category2id[category]

emotion_to_category_id[label2id["neutral"]] = category2id["neutral"]
NUM_LABELS = len(category_labels)

def encode_category(emotion_indices):
    first_emotion_index = int(str(emotion_indices).split(",")[0])
    return emotion_to_category_id[first_emotion_index]

for df in [train_df, valid_df, test_df]:
    df["category"] = df[LABEL_COLUMN].apply(encode_category)

print("\nCategory labels:")
print(category2id)




Category labels:
{'anger': 0, 'disgust': 1, 'fear': 2, 'joy': 3, 'sadness': 4, 'surprise': 5, 'neutral': 6}


In [9]:
"""
Train / validation / test split
"""

print("\nDataset sizes:")
print("Train:", len(train_df))
print("Validation:", len(valid_df))
print("Test:", len(test_df))




Dataset sizes:
Train: 43410
Validation: 5426
Test: 5427


In [10]:

"""
Tokenizer
"""

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, timeout=120)



In [11]:
"""
PyTorch Dataset
"""

class EmotionDataset(Dataset):

    def __init__(self, dataframe, tokenizer, max_length):
        self.texts = dataframe[TEXT_COLUMN].tolist()
        self.labels = dataframe["category"].astype(int).tolist()

        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, index):

        text = str(self.texts[index])
        label = self.labels[index]

        encoding = self.tokenizer(
            text,
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "labels": torch.tensor(label, dtype=torch.long)
        }


train_dataset = EmotionDataset(
    train_df,
    tokenizer,
    MAX_LENGTH
)

val_dataset = EmotionDataset(
    valid_df,
    tokenizer,
    MAX_LENGTH
)

test_dataset = EmotionDataset(
    test_df,
    tokenizer,
    MAX_LENGTH
)



In [12]:

"""
DataLoaders
"""

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False
)



In [13]:
"""
Initiating model
"""

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=NUM_LABELS,
    id2label=id2category,
    label2id=category2id,
    problem_type="single_label_classification"
)

model.to(DEVICE)

if NUM_GPUS > 1:
    model = torch.nn.DataParallel(
        model,
        device_ids=list(range(NUM_GPUS)),
        output_device=0
    )
    print(f"Using DataParallel across {NUM_GPUS} GPUs")


Some weights of XLMRobertaForSequenceClassification were not initialized from the model checkpoint at xlm-roberta-base and are newly initialized: ['classifier.dense.bias', 'classifier.dense.weight', 'classifier.out_proj.bias', 'classifier.out_proj.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [14]:
"""
Optimizer and scheduler
"""

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=LEARNING_RATE
)

total_training_steps = (
    len(train_loader) * EPOCHS
)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=int(0.1 * total_training_steps),
    num_training_steps=total_training_steps
)


In [15]:
"""
Focal loss
"""

class_counts = torch.bincount(
    torch.tensor(train_dataset.labels),
    minlength=NUM_LABELS
).to(DEVICE)
class_weights = class_counts.sum() / (
    NUM_LABELS * class_counts.clamp(min=1)
)


def focal_loss(logits, targets):
    cross_entropy = torch.nn.functional.cross_entropy(
        logits,
        targets,
        reduction="none"
    )
    probability_of_target = torch.exp(-cross_entropy)
    target_weights = class_weights[targets]

    return (
        target_weights
        * (1 - probability_of_target).pow(FOCAL_GAMMA)
        * cross_entropy
    ).mean()


In [16]:
"""
Evaluation function
"""

def evaluate(model, dataloader):

    model.eval()

    all_predictions = []
    all_labels = []

    total_loss = 0

    with torch.no_grad():

        for batch in dataloader:

            input_ids = batch["input_ids"].to(DEVICE)
            attention_mask = batch["attention_mask"].to(DEVICE)
            labels_batch = batch["labels"].to(DEVICE)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )
            loss = focal_loss(outputs.logits, labels_batch)

            total_loss += loss.item()
            predictions = outputs.logits.argmax(dim=-1)

            all_predictions.extend(predictions.cpu().tolist())
            all_labels.extend(labels_batch.cpu().tolist())

    average_loss = total_loss / len(dataloader)

    accuracy = accuracy_score(
        all_labels,
        all_predictions
    )

    balanced_accuracy = balanced_accuracy_score(
        all_labels,
        all_predictions
    )

    macro_f1 = f1_score(
        all_labels,
        all_predictions,
        average="macro",
        zero_division=0
    )

    weighted_f1 = f1_score(
        all_labels,
        all_predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "loss": average_loss,
        "accuracy": accuracy,
        "balanced_accuracy": balanced_accuracy,
        "macro_f1": macro_f1,
        "weighted_f1": weighted_f1,
        "labels": all_labels,
        "predictions": all_predictions
    }


In [17]:
"""
Train with focal loss
"""

best_model_path = str(MODELS_DIR / "best_goemotions_focal_model.pt")
best_validation_macro_f1 = -1.0

for epoch in range(EPOCHS):

    model.train()
    total_training_loss = 0

    for batch in train_loader:

        input_ids = batch["input_ids"].to(DEVICE)
        attention_mask = batch["attention_mask"].to(DEVICE)
        labels_batch = batch["labels"].to(DEVICE)

        optimizer.zero_grad(set_to_none=True)

        outputs = model(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        loss = focal_loss(outputs.logits, labels_batch)
        loss.backward()

        optimizer.step()
        scheduler.step()
        total_training_loss += loss.item()

    average_training_loss = total_training_loss / len(train_loader)
    validation_results = evaluate(model, val_loader)

    print(
        f"Epoch {epoch + 1}/{EPOCHS} | "
        f"Train focal loss: {average_training_loss:.4f} | "
        f"Validation focal loss: {validation_results['loss']:.4f} | "
        f"Validation macro-F1: {validation_results['macro_f1']:.4f}"
    )

    if validation_results["macro_f1"] > best_validation_macro_f1:
        best_validation_macro_f1 = validation_results["macro_f1"]
        checkpoint_model = model.module if isinstance(model, torch.nn.DataParallel) else model
        torch.save(checkpoint_model.state_dict(), best_model_path)


Epoch 1/4 | Train focal loss: 1.3324 | Validation focal loss: 0.8635 | Validation macro-F1: 0.4791


Epoch 2/4 | Train focal loss: 0.8286 | Validation focal loss: 0.7667 | Validation macro-F1: 0.5217


Epoch 3/4 | Train focal loss: 0.7030 | Validation focal loss: 0.7767 | Validation macro-F1: 0.5176


Epoch 4/4 | Train focal loss: 0.6347 | Validation focal loss: 0.7874 | Validation macro-F1: 0.5289


In [18]:

"""
Load best model
"""

model_state = torch.load(
    best_model_path,
    map_location=DEVICE
)

if isinstance(model, torch.nn.DataParallel):
    model.module.load_state_dict(model_state)
else:
    model.load_state_dict(model_state)


In [19]:

"""
Final test evaluation
"""

test_results = evaluate(
    model,
    test_loader
)

print("\n==============================")
print("FINAL E0 TEST RESULTS")
print("==============================")

print(
    f"Accuracy:     "
    f"{test_results['accuracy']:.4f}"
)

print(
    f"Balanced accuracy: "
    f"{test_results['balanced_accuracy']:.4f}"
)

print(
    f"Macro-F1:     "
    f"{test_results['macro_f1']:.4f}"
)

print(
    f"Weighted-F1:  "
    f"{test_results['weighted_f1']:.4f}"
)




FINAL E0 TEST RESULTS
Accuracy:     0.6051
Balanced accuracy: 0.6562
Macro-F1:     0.5310
Weighted-F1:  0.6089


In [20]:
"""
Per-category results
"""

print("\nClassification Report:")

print(
    classification_report(
        test_results["labels"],
        test_results["predictions"],
        labels=list(range(NUM_LABELS)),
        target_names=category_labels,
        digits=4,
        zero_division=0
    )
)




Classification Report:
              precision    recall  f1-score   support

       anger     0.4467    0.5306    0.4850       703
     disgust     0.2327    0.7619    0.3565        84
        fear     0.3956    0.8000    0.5294        90
         joy     0.8182    0.7537    0.7846      2054
     sadness     0.4057    0.6309    0.4938       317
    surprise     0.4738    0.7417    0.5782       573
     neutral     0.7057    0.3748    0.4896      1606

    accuracy                         0.6051      5427
   macro avg     0.4969    0.6562    0.5310      5427
weighted avg     0.6603    0.6051    0.6089      5427



In [21]:
"""
Confusion matrix
"""

category_confusion_matrix = confusion_matrix(
    test_results["labels"],
    test_results["predictions"],
    labels=list(range(NUM_LABELS))
)

print("\nCategory Confusion Matrix:")
print(
    pd.DataFrame(
        category_confusion_matrix,
        index=category_labels,
        columns=category_labels
    )
)




Category Confusion Matrix:
          anger  disgust  fear   joy  sadness  surprise  neutral
anger       373       95    14    45       62        57       57
disgust       8       64     4     3        2         3        0
fear          4        4    72     0       10         0        0
joy         101       26    32  1548       77       128      142
sadness      27       23    12    19      200        20       16
surprise     30       10     8    37       27       425       36
neutral     292       53    40   240      115       264      602
